# Chapter-10：提示词自进化

> Last updated: 2026-11-23

## 学习目标

第 8 章的多智能体系统里，任务拆解、依赖分析、智能体路由用的提示词都是事先固定好的模板，边界情况一多就容易出错。本章演示提示词自进化：先用一个基准（benchmark）给拆解和路由分别打分（L1、L2，都是 0 到 1 分），再从低分样例出发，用 TextGrad 的文本梯度自动优化提示词，最后在开发集（dev）上复评，只有不退化才保留（rollback）。重点理解三件事：

1. 基准评测：用 `fixtures.json` 里 15 条旅行用例逐条对比期望，得到 L1（拆解）、L2（路由）两个 0 到 1 分。
2. 两种 TextGrad 用法：`direct`（单变量，loss 直接批评 prompt）与 `planner_graph`（把拆解→依赖→路由建成可反传计算图）。
3. rollback：候选提示词在 dev 上不优于当前最优时拒绝，保证单调不退化。

## TextGrad 是什么

TextGrad 是一种以文本为对象、以语言模型为引擎的自动优化方法，核心思想可概括为“通过文本进行自动微分”。它将提示词、回答等文本对象视为可微分变量，先由语言模型依据评测反馈生成一段自然语言的梯度——即指出文本当前不足与改进方向的文字——再沿该梯度更新变量，如此迭代直至评测指标不再提升。

本章主要使用它的两个核心原语：`TextLoss`（把评测反馈转化为损失）与 `TextualGradientDescent`（沿文本梯度更新变量）。其价值在于，将原本依赖人工反复试写的提示词迭代，转化为由评测分数驱动、可回滚的自动化过程。

## 两种模式速览

| 模式 | backend 参数 | 机制 | 适合 |
|------|------|------|------|
| direct | `"direct"` | 只把 prompt 当作可训练变量，loss 让 LLM 直接批评 prompt + 失败样例，再 `TextualGradientDescent` 更新 | 快速、低成本地改进提示词 |
| planner_graph | `"planner_graph"` | 把拆解→依赖→路由整条流水线用 `StringBasedFunction` 建成计算图，loss 比较流水线输出与期望，文本梯度穿过真实步骤回传 | 希望梯度真正流经整条规划流水线 |

## 自进化流程

整个自进化是一个循环：train 负责发现失败样例并生成文本梯度，dev 负责验收候选并决定接受或回滚，test 留作最终评测（本次不参与）。流程如下：

```mermaid
flowchart LR
  A[默认提示词] --> B[在 dev 上计算基线 L1/L2]
  B --> C{train 是否还有失败样例}
  C -->|是| D[依据失败样例生成文本梯度]
  D --> E[更新为候选提示词]
  E --> F[在 dev 上复评候选]
  F --> G{候选是否不退化}
  G -->|是| H[接受候选]
  G -->|否| I[回滚，保留原提示词]
  H --> C
  I --> C
  C -->|否| J[结束：输出最终提示词]
```

## 1. 环境准备与模块导入

本章复用第 8 章的 `travel_multi_agent` 包与仓库根 `llm.py`，并需要已安装 `textgrad`（见 `README.md`）。下面把 chapter10、Chapter-8 和仓库根加入 `sys.path`，再导入 LLM、评测、优化入口与注册表。

In [1]:
import sys
from pathlib import Path

_CHAPTER10 = Path.cwd().resolve()
if not (_CHAPTER10 / 'optimize.py').is_file():
    _CHAPTER10 = _CHAPTER10 / 'chapter10'
if not (_CHAPTER10 / 'optimize.py').is_file():
    raise FileNotFoundError('请在 chapter10 目录或书仓库根目录运行本 Notebook')

_BOOK_ROOT = _CHAPTER10.parent
_CHAPTER8 = _BOOK_ROOT / 'Chapter-8'
for _p in (_CHAPTER10, _CHAPTER8, _BOOK_ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from llm import create_llm
from optimizable_prompts import (
    AGENT_ROUTING_PROMPT,
    TASK_DECOMPOSITION_PROMPT,
    DECOMPOSITION_PLACEHOLDERS,
    ROUTING_PLACEHOLDERS,
)
from evaluate import evaluate_benchmark, evaluate_case, load_fixtures
from optimize import SlotSpec, run_planner_optimization, save_planner_optimization_artifacts
from runtime import Planner
from travel_multi_agent.agents.registry import create_travel_registry

FIXTURES = load_fixtures(_CHAPTER10 / 'fixtures.json')
REGISTRY = create_travel_registry()

EXECUTOR_LLM = create_llm(temperature=0)
OPTIMIZER_LLM = create_llm(temperature=0.2)

SLOTS = [
    SlotSpec('decomposition_prompt', DECOMPOSITION_PLACEHOLDERS, metric='l1'),
    SlotSpec('agent_routing', ROUTING_PLACEHOLDERS, metric='l2'),
]


def make_planner(decomposition_prompt, agent_routing):
    return Planner(
        EXECUTOR_LLM,
        REGISTRY,
        decomposition_prompt=decomposition_prompt,
        agent_routing=agent_routing,
    )


print('chapter10 就绪：cases =', len(FIXTURES['cases']))

chapter10 就绪：cases = 33


## 2. 基准评测与打分

基准数据由开发者预先构建，`fixtures.json` 收录旅行用例，每条包含
- 用户问题 `query` 
- 期望字段：`expect` 声明最少/最多子任务数、必须保留的关键词、禁止出现的词、可匹配的 Agent 及工具参数校验，
- `expect_routing` 声明每个子任务应路由到的 Agent。
这些用例按用途切分为 train 条、dev 条、test 条：
- train 用于发现低分样例并生成文本梯度，
- dev 用于验收候选提示词是否提升，
- test 保留作最终评测。

评测采用规则打分并归一化到 0 到 1 分：L1（拆解）由格式 0.10、子任务数量 0.15、槽位齐全 0.25、无禁止词 0.10 组成；L2（路由）由 Agent 匹配 0.25、Agent 覆盖 0.15、参数完整 0.20 组成。下面先打印划分，再对一条用例执行一次完整打分，仅用于展示分数由哪些子项构成；后续第 5、6 节的实际优化会对 train 的全部 7 条样本执行，并用 dev 的 2 条验收。

In [2]:
print('=== fixtures.json 划分 ===')
print('total cases =', len(FIXTURES['cases']))
for split in ('train', 'dev', 'test'):
    case_ids = FIXTURES['splits'][split]
    print(f'{split}: {len(case_ids)} 条 -> {case_ids}')

print()
sample_case = FIXTURES['cases'][0]
print(f"样例 {sample_case['id']}: query={sample_case['query']}")

result = await evaluate_case(
    make_planner(TASK_DECOMPOSITION_PROMPT, AGENT_ROUTING_PROMPT),
    sample_case,
    metric='l1_l2',
)
print(f'L1 拆解 = {result.l1:.3f}  |  L2 路由 = {result.l2:.3f}  |  总分 = {result.total:.3f}')
print('L1 子项：格式', result.format_ok, '| 子任务数', result.subtask_count_ok, '| 槽位', result.slots_ok, '| 禁止词', result.forbidden_ok)
print('L2 子项：匹配', result.routing_ok, '| 覆盖', result.agent_coverage_ok, '| 参数', result.params_ok)
print('失败原因（details）:', result.details)

=== fixtures.json 划分 ===
total cases = 33
train: 18 条 -> ['case-01', 'case-02', 'case-03', 'case-04', 'case-05', 'case-06', 'case-07', 'case-16', 'case-17', 'case-18', 'case-19', 'case-20', 'case-24', 'case-25', 'case-26', 'case-27', 'case-28', 'case-29']
dev: 6 条 -> ['case-08', 'case-09', 'case-21', 'case-22', 'case-30', 'case-31']
test: 9 条 -> ['case-10', 'case-11', 'case-12', 'case-13', 'case-14', 'case-15', 'case-23', 'case-32', 'case-33']

样例 case-01: query=帮我查一下北京明天天气怎么样
L1 拆解 = 1.000  |  L2 路由 = 1.000  |  总分 = 1.000
L1 子项：格式 True | 子任务数 True | 槽位 True | 禁止词 True
L2 子项：匹配 True | 覆盖 True | 参数 True
失败原因（details）: []


## 3. 基线评测（L1 拆解 + L2 路由）

先不执行优化，用第 8 章的默认提示词在 dev 上执行一次评测，得到 L1 / L2 的基线分：

- `metric='l1'`：只评测任务拆解。
- `metric='l2'`：只评测智能体路由。

优化前后都看这两个分，才能判断提示词是否真的变好。

In [3]:
baseline_planner = make_planner(TASK_DECOMPOSITION_PROMPT, AGENT_ROUTING_PROMPT)
baseline_l1 = await evaluate_benchmark(baseline_planner, FIXTURES, 'dev', metric='l1')
baseline_l2 = await evaluate_benchmark(baseline_planner, FIXTURES, 'dev', metric='l2')

print('=== 基线（优化前，dev）===')
print(f'L1 decomposition: {baseline_l1.average_score:.3f} cases={baseline_l1.case_count}')
for r in baseline_l1.cases:
    status = 'PASS' if r.total >= 0.8 else 'FAIL'
    print(f'  [{status}] {r.case_id} score={r.total:.3f} subtasks={r.subtask_count}')
print(f'L2 routing: {baseline_l2.average_score:.3f} cases={baseline_l2.case_count}')
for r in baseline_l2.cases:
    status = 'PASS' if r.total >= 0.8 else 'FAIL'
    print(f'  [{status}] {r.case_id} score={r.total:.3f}')
    for d in r.details:
        print(f'       - {d}')

=== 基线（优化前，dev）===
L1 decomposition: 0.847 cases=6
  [PASS] case-08 score=1.000 subtasks=3
  [PASS] case-09 score=1.000 subtasks=2
  [FAIL] case-21 score=0.750 subtasks=3
  [PASS] case-22 score=0.833 subtasks=4
  [FAIL] case-30 score=0.750 subtasks=3
  [FAIL] case-31 score=0.750 subtasks=5
L2 routing: 1.000 cases=6
  [PASS] case-08 score=1.000
  [PASS] case-09 score=1.000
  [PASS] case-21 score=1.000
  [PASS] case-22 score=1.000
  [PASS] case-30 score=1.000
  [PASS] case-31 score=1.000


## 4. 自进化流程的分步拆解

下面把一轮自进化循环分步拆解：先在所有 train 样本中查找失败样例，再依据失败样例生成文本梯度得到候选提示词，最后在 dev 上复评并决定回滚。`run_planner_optimization`（第 5、6 节使用）正是把这套流程自动化后的入口，其内部逻辑与本 cell 一致。

In [4]:
from evaluate import collect_failures
from optimizable_prompts import extract_prompt_template
from optimize import make_direct_proposer

planner = make_planner(TASK_DECOMPOSITION_PROMPT, AGENT_ROUTING_PROMPT)
train_cases = [c for c in FIXTURES['cases'] if c['id'] in FIXTURES['splits']['train']]

# ① 在 dev 上计算基线得分
baseline = await evaluate_benchmark(planner, FIXTURES, 'dev', metric='l1')
print('① dev 基线 L1 =', round(baseline.average_score, 3))

# ② 在全部 train 样本中查找失败样例
failures = await collect_failures(planner, train_cases, failure_threshold=0.99, metric='l1')
print('\n② train 失败样例 =', len(failures), '条')
for score, case in failures:
    print('   ', case['id'], 'L1=', round(score.total, 3), '|', '; '.join(score.details) or '—')

# ③ 依据失败样例生成文本梯度，得到候选提示词
propose = make_direct_proposer(OPTIMIZER_LLM)
new_decomposition, new_routing = propose(
    TASK_DECOMPOSITION_PROMPT, AGENT_ROUTING_PROMPT, 'decomposition_prompt', failures
)
print('\n③ 候选拆解提示词：')
print(new_decomposition)

# 校验候选是否仍为合法模板
try:
    new_decomposition = extract_prompt_template(
        new_decomposition, DECOMPOSITION_PLACEHOLDERS
    )
except ValueError as exc:
    print('\n④ 候选模板不合法，直接回滚：', exc)
    print('⑤ 决策：回滚，保留原提示词')
else:
    # ④ 在 dev 上复评候选提示词
    candidate_report = await evaluate_benchmark(
        make_planner(new_decomposition, new_routing), FIXTURES, 'dev', metric='l1'
    )
    print('\n④ 候选 dev L1 =', round(candidate_report.average_score, 3))

    # ⑤ 依据评分执行回滚决策
    accepted = candidate_report.average_score >= baseline.average_score
    print('\n⑤ 决策：', '接受候选' if accepted else '回滚，保留原提示词')

① dev 基线 L1 = 0.806

② train 失败样例 = 5 条
    case-06 L1= 0.583 | 子任务数 5 不在 [1, 4]; 出现禁止词: ['航班']
    case-07 L1= 0.75 | 子任务数 2 不在 [1, 1]
    case-19 L1= 0.833 | 出现禁止词: ['酒店']
    case-20 L1= 0.75 | 子任务数 2 不在 [3, 4]
    case-26 L1= 0.75 | 子任务数 2 不在 [1, 1]

③ 候选拆解提示词：
你是专业的旅行规划与任务拆解助手。请根据用户提供的背景信息、可用能力列表以及具体需求，将复杂的旅行目标拆解为可执行的具体子任务。

背景信息：
{background_info}

可用能力：
{capability_catalog}

用户输入：
{user_input}

请严格遵循以下逻辑进行拆解：
1. 分析用户的核心意图和约束条件。
2. 结合可用能力，确定实现目标所需的关键步骤。
3. 确保子任务之间逻辑连贯、覆盖全面且具备可操作性。

请按以下格式输出：
# 目标
用一句话清晰概括最终要达成的旅行目标

# 任务拆解
- [步骤1] 子任务描述（明确动作与预期结果）
- [步骤2] 子任务描述
- [步骤3] 子任务描述
...

④ 候选 dev L1 = 0.806

⑤ 决策： 接受候选


## 5. 用 direct 模式优化（单变量文本梯度）

`direct` 模式把当前提示词当作可训练变量，用失败样例生成文本梯度再更新它。下面的调用会按 `decomposition_prompt → agent_routing` 的顺序各优化一次，并在每个槽位上执行 dev 复评与 rollback。执行完成后打印每一轮的接受/拒绝与最终保留的提示词。

In [5]:
direct = await run_planner_optimization(
    backend='direct',  # TextGrad 模式：direct / planner_graph
    slots=SLOTS,  # 要优化的槽位：decomposition_prompt(L1)、agent_routing(L2)
    decomposition_prompt=TASK_DECOMPOSITION_PROMPT,  # 拆解槽位初始提示词
    agent_routing=AGENT_ROUTING_PROMPT,  # 路由槽位初始提示词
    executor_llm=EXECUTOR_LLM,  # 执行拆解/依赖/路由并打分的模型
    optimizer_llm=OPTIMIZER_LLM,  # 生成文本梯度、改写提示词的模型
    registry=REGISTRY,  # 领域智能体注册表（能力目录 + Agent 参数）
    fixtures=FIXTURES,  # 基准数据（15 条用例 + train/dev/test 切分）
    max_steps=1,  # 每个槽位最多优化几轮
    failure_threshold=0.9,  # 得分低于该值的样例视为失败
    rollback=True,  # 候选不优于当前最优时是否拒绝
    train_split='train',  # 找失败样例的集合（train，7 条）
    dev_split='dev',  # 验收候选的集合（dev，2 条）
    objective='l1_l2',  # 优化目标：同时看 L1 和 L2
)

for result in direct['results']:
    print(f'{result.slot}: 基线 {result.baseline_score:.3f} -> 最优 {result.best_score:.3f}')
    for s in result.steps:
        decision = '接受' if s.accepted else '拒绝'
        print(f'  第 {s.step} 轮：{decision}，候选 dev={s.candidate_score:.3f}，失败样例={s.failure_count} 条')

print()
for result in direct['results']:
    print(f'--- {result.slot} 最终保留的提示词（前 400 字）---')
    print(result.best_prompt[:400])
    print()

decomposition_prompt: 基线 0.847 -> 最优 0.847
  第 1 轮：拒绝，候选 dev=0.806，失败样例=3 条
agent_routing: 基线 1.000 -> 最优 1.000
  第 1 轮：拒绝，候选 dev=1.000，失败样例=0 条

--- decomposition_prompt 最终保留的提示词（前 400 字）---
你是任务拆解助手，请根据用户输入拆解任务。

背景信息：
{background_info}

可用能力：
{capability_catalog}

用户输入：
{user_input}

请按以下格式输出：
# 目标
一句话概括目标
# 任务拆解
- 子任务1
- 子任务2


--- agent_routing 最终保留的提示词（前 400 字）---
为每个子任务选择一个智能体，并输出 JSON 数组。

智能体列表：
{agent_team}

事实：
{pre_survey_json}

子任务：
{subtasks_json}

依赖：
{depends_on_json}

只输出 JSON 数组，格式如下：
[{{"task_id": "T1", "agent": "智能体名", "params": {{}}}}]




## 6. 用 planner_graph 模式优化（流水线计算图）

与 direct 的差别只在 `backend='planner_graph'`：它把拆解→依赖→路由用 `StringBasedFunction` 串成计算图，候选提示词由整条流水线的反向传播得到。其余参数与 direct 一节一致。

In [6]:
planner = await run_planner_optimization(
    backend='planner_graph',  # 参数含义与 direct 一节相同，仅后端不同
    slots=SLOTS,
    decomposition_prompt=TASK_DECOMPOSITION_PROMPT,
    agent_routing=AGENT_ROUTING_PROMPT,
    executor_llm=EXECUTOR_LLM,
    optimizer_llm=OPTIMIZER_LLM,
    registry=REGISTRY,
    fixtures=FIXTURES,
    max_steps=1,
    failure_threshold=0.9,
    rollback=True,
    train_split='train',
    dev_split='dev',
    objective='l1_l2',
)

for result in planner['results']:
    print(f'{result.slot}: 基线 {result.baseline_score:.3f} -> 最优 {result.best_score:.3f}')
    for s in result.steps:
        decision = '接受' if s.accepted else '拒绝'
        print(f'  第 {s.step} 轮：{decision}，候选 dev={s.candidate_score:.3f}，失败样例={s.failure_count} 条')

print()
for result in planner['results']:
    print(f'--- {result.slot} 最终保留的提示词（前 400 字）---')
    print(result.best_prompt[:400])
    print()

decomposition_prompt: 基线 0.806 -> 最优 0.806
  第 1 轮：拒绝，候选 dev=0.806，失败样例=3 条
agent_routing: 基线 1.000 -> 最优 1.000
  第 1 轮：拒绝，候选 dev=1.000，失败样例=0 条

--- decomposition_prompt 最终保留的提示词（前 400 字）---
你是任务拆解助手，请根据用户输入拆解任务。

背景信息：
{background_info}

可用能力：
{capability_catalog}

用户输入：
{user_input}

请按以下格式输出：
# 目标
一句话概括目标
# 任务拆解
- 子任务1
- 子任务2


--- agent_routing 最终保留的提示词（前 400 字）---
为每个子任务选择一个智能体，并输出 JSON 数组。

智能体列表：
{agent_team}

事实：
{pre_survey_json}

子任务：
{subtasks_json}

依赖：
{depends_on_json}

只输出 JSON 数组，格式如下：
[{{"task_id": "T1", "agent": "智能体名", "params": {{}}}}]




## 7. 保存产物并复评对比

把两种模式最终保留的提示词和逐轮报告写入 `output/<backend>/`，再从文件重新加载提示词执行 L1 / L2 复评，与基线对比，验证保存与加载的闭环。

In [7]:
import json

for backend_name, outcome in (('direct', direct), ('planner_graph', planner)):
    output_dir = _CHAPTER10 / 'output' / backend_name
    prompts_path, report_path = save_planner_optimization_artifacts(
        prompts=outcome['prompts'],
        results=outcome['results'],
        output_dir=output_dir,
        objective='l1_l2',
        train_split='train',
        dev_split='dev',
    )

    saved_prompts = json.loads(prompts_path.read_text(encoding='utf-8'))
    after_planner = make_planner(
        saved_prompts['decomposition_prompt'],
        saved_prompts['agent_routing'],
    )
    after_l1 = await evaluate_benchmark(after_planner, FIXTURES, 'dev', metric='l1')
    after_l2 = await evaluate_benchmark(after_planner, FIXTURES, 'dev', metric='l2')

    print(f'=== {backend_name} 对比（dev，从文件重新加载）===')
    print(f'L1 decomposition: {baseline_l1.average_score:.3f} -> {after_l1.average_score:.3f}')
    print(f'L2 routing:       {baseline_l2.average_score:.3f} -> {after_l2.average_score:.3f}')
    print(f'已保存 prompt: {prompts_path}')
    print(f'已保存报告:   {report_path}')
    print()

=== direct 对比（dev，从文件重新加载）===
L1 decomposition: 0.847 -> 0.847
L2 routing:       1.000 -> 1.000
已保存 prompt: E:\Book\agent-systems-book\chapter10\output\direct\optimized.json
已保存报告:   E:\Book\agent-systems-book\chapter10\output\direct\optimization_report.json

=== planner_graph 对比（dev，从文件重新加载）===
L1 decomposition: 0.847 -> 0.847
L2 routing:       1.000 -> 1.000
已保存 prompt: E:\Book\agent-systems-book\chapter10\output\planner_graph\optimized.json
已保存报告:   E:\Book\agent-systems-book\chapter10\output\planner_graph\optimization_report.json



## 小结

本章把提示词优化转化为一个可自动迭代的闭环：优化对象收敛为带占位符的提示词模板，以任务拆解（L1）与智能体路由（L2）两个分数作为反馈信号，并通过回滚保证每一轮都不比当前最优版本更差。两种实现路径各有侧重：单变量方式（direct）结构简洁、开销较小，仅把提示词视为唯一可优化的文本；计算图方式（planner_graph）则将拆解、依赖分析、路由三步串联为可反向传播的计算图，使文本梯度流经整条规划流水线，反馈信号更贴近真实执行过程。